[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmg777/starter-academic-v501/blob/master/content/post/python_esda2/notebook.ipynb)

# Exploratory Spatial Data Analysis: Spatial Clusters and Dynamics of Human Development in South America

When we look at a map of human development across South America, a pattern immediately stands out: prosperous regions tend to cluster together, and so do lagging regions. But is this clustering statistically significant, or could it arise by chance? And how have these spatial clusters evolved over time?

**Exploratory Spatial Data Analysis (ESDA)** provides the tools to answer these questions. ESDA is a set of techniques for visualizing spatial distributions, identifying patterns of spatial clustering, and detecting spatial outliers. Unlike standard exploratory data analysis, which treats observations as independent, ESDA explicitly accounts for the geographic location of each observation and the relationships between neighbors.

This tutorial uses the [Subnational Human Development Index](https://globaldatalab.org/shdi/) (SHDI) from [Smits and Permanyer (2019)](https://doi.org/10.1038/sdata.2019.38) for **153 sub-national regions across 12 South American countries** in 2013 and 2019. We progress from simple scatter plots and choropleth maps to formal tests of spatial dependence (Moran's I), local cluster identification (LISA maps), and space-time dynamics. By the end, you will be able to answer: **do nearby regions in South America share similar development levels, and how have these spatial clusters evolved between 2013 and 2019?**

**Learning objectives:**

- Understand the concept of spatial autocorrelation and why it matters for regional analysis
- Create choropleth maps and scatter plots to visualize spatial distributions
- Build and interpret a spatial weights matrix using Queen contiguity
- Compute and interpret global Moran's I for spatial dependence testing
- Identify local spatial clusters (HH, LL) and outliers (HL, LH) using LISA statistics
- Explore space-time dynamics of spatial clusters using directional Moran scatter plots
- Compare country-level development trajectories within the spatial framework

## 2. The ESDA pipeline

The analysis follows a natural progression: **Step 1** Load & Explore --> **Step 2** Visualize Maps --> **Step 3** Spatial Weights --> **Step 4** Global Moran's I --> **Step 5** Local LISA --> **Step 6** Space-Time Dynamics

Steps 1--2 are purely visual --- they build intuition about where high and low values are concentrated. Step 3 formalizes the notion of "neighbors" through a spatial weights matrix. Steps 4--5 use that matrix to compute statistics that quantify spatial clustering, first globally (one number for the whole map) and then locally (one number per region). Step 6 connects the spatial and temporal dimensions by tracking how regions move through the Moran scatter plot between periods.

## 3. Setup and imports

The analysis uses [GeoPandas](https://geopandas.org/) for spatial data handling, [PySAL](https://pysal.org/) for spatial statistics, and [splot](https://splot.readthedocs.io/) for specialized spatial visualizations.

In [ ]:
!pip install -q geopandas libpysal esda splot mapclassify adjustText

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from libpysal.weights import Queen
from libpysal.weights import lag_spatial
from esda.moran import Moran, Moran_Local
from splot.esda import moran_scatterplot, lisa_cluster
from splot.libpysal import plot_spatial_weights
from adjustText import adjust_text
import mapclassify

# Reproducibility
RANDOM_SEED = 42

# Site color palette
STEEL_BLUE = "#6a9bcc"
WARM_ORANGE = "#d97757"
NEAR_BLACK = "#141413"
TEAL = "#00d4c8"

In [ ]:
# Dark theme palette
DARK_NAVY = "#0f1729"
GRID_LINE = "#1f2b5e"
LIGHT_TEXT = "#c8d0e0"
WHITE_TEXT = "#e8ecf2"

plt.rcParams.update({
    "figure.facecolor": DARK_NAVY,
    "axes.facecolor": DARK_NAVY,
    "axes.edgecolor": DARK_NAVY,
    "axes.linewidth": 0,
    "axes.labelcolor": LIGHT_TEXT,
    "axes.titlecolor": WHITE_TEXT,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.spines.left": False,
    "axes.spines.bottom": False,
    "axes.grid": True,
    "grid.color": GRID_LINE,
    "grid.linewidth": 0.6,
    "grid.alpha": 0.8,
    "xtick.color": LIGHT_TEXT,
    "ytick.color": LIGHT_TEXT,
    "xtick.major.size": 0,
    "ytick.major.size": 0,
    "text.color": WHITE_TEXT,
    "font.size": 12,
    "legend.frameon": False,
    "legend.fontsize": 11,
    "legend.labelcolor": LIGHT_TEXT,
    "figure.edgecolor": DARK_NAVY,
    "savefig.facecolor": DARK_NAVY,
    "savefig.edgecolor": DARK_NAVY,
})

## 4. Data loading and exploration

The dataset is a GeoJSON file containing polygon geometries and development indicators for 153 sub-national regions across South America, sourced from the [Global Data Lab](https://globaldatalab.org/shdi/) ([Smits and Permanyer, 2019](https://doi.org/10.1038/sdata.2019.38)). Each region has the Subnational Human Development Index (SHDI) and its three component indices --- Health, Education, and Income --- for 2013 and 2019.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/cmg777/starter-academic-v501/master/content/post/python_esda2/data.geojson"
gdf = gpd.read_file(DATA_URL)
print(f"Loaded: {gdf.shape[0]} rows, {gdf.shape[1]} columns")
print(f"Countries: {gdf['country'].nunique()}")
print(f"CRS: {gdf.crs}")

Before computing change columns, we prepare the data for labeling. Some region names in the raw data are very long, so we simplify them. We also create a `region_country` column that appends the ISO country code to each region name.

In [ ]:
COUNTRY_ISO = {
    "Argentina": "ARG", "Bolivia": "BOL", "Brazil": "BRA",
    "Chili": "CHL", "Colombia": "COL", "Ecuador": "ECU",
    "Guyana": "GUY", "Paraguay": "PRY", "Peru": "PER",
    "Suriname": "SUR", "Uruguay": "URY", "Venezuela": "VEN",
}
gdf["country_iso"] = gdf["country"].map(COUNTRY_ISO)

RENAME = {
    "Catamarca, La Rioja, San Juan": "Catamarca-La Rioja",
    "Corrientes, Entre Rios, Misiones": "Corrientes-Misiones",
    "Chubut, Neuquen, Rio Negro, Santa Cruz, Tierra del Fuego": "Patagonia",
    "La Pampa, San Luis, Mendoza": "La Pampa-Mendoza",
    "Santiago del Estero, Tucuman": "Tucuman-Sgo Estero",
    "Tarapaca (incl Arica and Parinacota)": "Tarapaca",
    "Valparaiso (former Aconcagua)": "Valparaiso",
    "Los Lagos (incl Los Rios)": "Los Lagos",
    "Magallanes and La Antartica Chilena": "Magallanes",
    "Antioquia (incl Medellin)": "Antioquia",
    "Atlantico (incl Barranquilla)": "Atlantico",
    "Bolivar (Sur and Norte)": "Bolivar",
    "Essequibo Islands-West Demerara": "Essequibo-W Demerara",
    "East Berbice-Corentyne": "E Berbice-Corentyne",
    "Upper Takutu-Upper Essequibo": "Upper Takutu-Essequibo",
    "Upper Demerara-Berbice": "Upper Demerara",
    "Cuyuni-Mazaruni-Upper Essequibo": "Cuyuni-Mazaruni",
    "Region Metropolitana": "R. Metropolitana",
    "Federal District": "Federal Dist.",
    "City of Buenos Aires": "C. Buenos Aires",
    "Brokopondo and Sipaliwini": "Brokopondo-Sipaliwini",
    "Montevideo and Metropolitan area": "Montevideo",
}
gdf["region"] = gdf["region"].replace(RENAME)
gdf["region_country"] = gdf["region"] + " (" + gdf["country_iso"] + ")"

We then compute the change in SHDI and its components between the two periods.

In [ ]:
gdf["shdi_change"] = gdf["shdi2019"] - gdf["shdi2013"]
gdf["health_change"] = gdf["healthindex2019"] - gdf["healthindex2013"]
gdf["educ_change"] = gdf["edindex2019"] - gdf["edindex2013"]
gdf["income_change"] = gdf["incindex2019"] - gdf["incindex2013"]

print(gdf[["shdi2013", "shdi2019", "shdi_change"]].describe().round(4).to_string())

The dataset covers 153 regions across 12 South American countries. Mean SHDI increased modestly from 0.7424 in 2013 to 0.7477 in 2019 (+0.0053), but the change varied widely: from a maximum decline of -0.0670 to a maximum improvement of +0.0450. The standard deviation of SHDI also increased slightly (0.0594 to 0.0613), hinting that regional disparities may have widened.

## 5. Exploratory scatter plots

### 5.1 HDI scatter: 2013 vs 2019

A scatter plot of SHDI in 2013 against SHDI in 2019 provides a quick overview of temporal dynamics. Points above the 45-degree line represent regions that improved; points below represent regions that declined.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

ax.scatter(gdf["shdi2013"], gdf["shdi2019"],
           color=STEEL_BLUE, edgecolors=DARK_NAVY, s=45, alpha=0.75, zorder=3)

lims = [min(gdf["shdi2013"].min(), gdf["shdi2019"].min()) - 0.01,
        max(gdf["shdi2013"].max(), gdf["shdi2019"].max()) + 0.01]
ax.plot(lims, lims, color=WARM_ORANGE, linewidth=1.5, linestyle="--",
        label="45\u00b0 line (no change)", zorder=2)

ax.set_xlabel("SHDI 2013")
ax.set_ylabel("SHDI 2019")
ax.set_title("Subnational HDI: 2013 vs 2019")
ax.legend()

residual = gdf["shdi2019"] - gdf["shdi2013"]
extremes = set()
extremes.update(residual.nlargest(3).index.tolist())
extremes.update(residual.nsmallest(3).index.tolist())
extremes.update(gdf["shdi2019"].nlargest(2).index.tolist())
extremes.update(gdf["shdi2019"].nsmallest(2).index.tolist())

texts = []
for i in extremes:
    texts.append(ax.text(gdf.loc[i, "shdi2013"], gdf.loc[i, "shdi2019"],
                         gdf.loc[i, "region_country"], fontsize=8, color=LIGHT_TEXT))
adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="-", color=LIGHT_TEXT,
            alpha=0.5, lw=0.5))

plt.show()

Of 153 regions, **126 improved** their SHDI between 2013 and 2019, while **27 declined**. The biggest decliners --- Federal Dist. (VEN), Carabobo (VEN), and Aragua (VEN) --- are all Venezuelan states. The biggest improvers --- Meta (COL), Vichada (COL), and Brokopondo-Sipaliwini (SUR) --- rose above the line, with gains up to +0.045 points.

### 5.2 Component scatter plots

The SHDI is a composite of three sub-indices: Health, Education, and Income. Breaking down the change by component reveals which dimensions drove the aggregate patterns.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

components = [
    ("healthindex2013", "healthindex2019", "Health Index"),
    ("edindex2013", "edindex2019", "Education Index"),
    ("incindex2013", "incindex2019", "Income Index"),
]

for ax, (col13, col19, label) in zip(axes, components):
    ax.scatter(gdf[col13], gdf[col19],
               color=STEEL_BLUE, edgecolors=DARK_NAVY, s=40, alpha=0.7, zorder=3)
    lims = [min(gdf[col13].min(), gdf[col19].min()) - 0.02,
            max(gdf[col13].max(), gdf[col19].max()) + 0.02]
    ax.plot(lims, lims, color=WARM_ORANGE, linewidth=1.5, linestyle="--", zorder=2)
    ax.set_xlabel(f"{label} 2013")
    ax.set_ylabel(f"{label} 2019")
    ax.set_title(label)

    comp_residual = gdf[col19] - gdf[col13]
    comp_extremes = set()
    comp_extremes.update(comp_residual.nlargest(2).index.tolist())
    comp_extremes.update(comp_residual.nsmallest(2).index.tolist())
    texts = []
    for i in comp_extremes:
        texts.append(ax.text(gdf.loc[i, col13], gdf.loc[i, col19],
                             gdf.loc[i, "region_country"], fontsize=7, color=LIGHT_TEXT))
    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="-", color=LIGHT_TEXT,
                alpha=0.5, lw=0.5))

fig.suptitle("HDI components: 2013 vs 2019", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

The three components tell very different stories. Health and Education improved almost universally. Income, however, tells a starkly different story: **71 of 153 regions (46.4%) experienced a decline** in their income index between 2013 and 2019.

## 6. Choropleth maps

### 6.1 HDI levels across South America

Choropleth maps add the geographic dimension. We use Fisher-Jenks natural breaks computed from 2013 and held constant for 2019. Fisher-Jenks is a classification method that finds natural groupings in data by minimizing within-class variance.

In [ ]:
import mapclassify
from matplotlib.patches import Patch

fj = mapclassify.FisherJenks(gdf["shdi2013"].values, k=5)
breaks = fj.bins.tolist()

max_val = max(gdf["shdi2013"].max(), gdf["shdi2019"].max())
if max_val > breaks[-1]:
    breaks[-1] = float(round(max_val + 0.001, 3))

fj_2019 = mapclassify.UserDefined(gdf["shdi2019"].values, bins=breaks)

classes_2013 = fj.yb
classes_2019 = fj_2019.yb
improved = (classes_2019 > classes_2013).sum()
stayed = (classes_2019 == classes_2013).sum()
declined = (classes_2019 < classes_2013).sum()

print(f"Breaks (from 2013): {[round(b, 3) for b in breaks]}")
print(f"  Improved (moved up):   {improved}")
print(f"  Stayed same:           {stayed}")
print(f"  Declined (moved down): {declined}")

In [ ]:
class_labels = []
lower = round(gdf["shdi2013"].min(), 2)
for b in breaks:
    class_labels.append(f"{lower:.2f} \u2013 {b:.2f}")
    lower = round(b, 2)

fig, axes = plt.subplots(1, 2, figsize=(16, 12))
cmap = plt.cm.coolwarm
norm = plt.Normalize(vmin=0, vmax=len(breaks) - 1)

for ax, year_col, title, year_fj in [
    (axes[0], "shdi2013", "SHDI 2013", fj),
    (axes[1], "shdi2019", "SHDI 2019", fj_2019),
]:
    colors = [cmap(norm(c)) for c in year_fj.yb]
    gdf.plot(ax=ax, color=colors, edgecolor=GRID_LINE, linewidth=0.3)
    ax.set_title(title, fontsize=14, pad=10)
    ax.set_axis_off()

    counts = np.bincount(year_fj.yb, minlength=len(breaks))
    handles = [Patch(facecolor=cmap(norm(i)), edgecolor=GRID_LINE,
               label=f"{cl}  (n={c})")
               for i, (cl, c) in enumerate(zip(class_labels, counts))]
    ax.legend(handles=handles, title="SHDI Class", loc="lower right",
              fontsize=10, title_fontsize=11)

map_extremes = gdf["shdi2019"].nlargest(3).index.tolist() + \
               gdf["shdi2019"].nsmallest(3).index.tolist()
for ax_map in axes:
    texts = []
    for i in map_extremes:
        centroid = gdf.geometry.iloc[i].centroid
        texts.append(ax_map.text(centroid.x, centroid.y,
                     gdf.loc[i, "region_country"],
                     fontsize=7, color=WHITE_TEXT, weight="bold"))
    adjust_text(texts, ax=ax_map, arrowprops=dict(arrowstyle="-|>",
                color=LIGHT_TEXT, alpha=0.9, lw=1.2, mutation_scale=8))

plt.show()

**43 regions moved up** at least one class between 2013 and 2019, **86 stayed** in the same class, and **24 declined**. The Southern Cone and southern Brazil consistently occupy the highest class, while the Amazon basin, Guyana, and parts of Venezuela anchor the lowest class.

### 6.2 Mapping HDI change

A map of SHDI change (2019 minus 2013) reveals the geographic distribution of gains and losses, using a diverging color scale centered at zero.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 10))

abs_max = max(abs(gdf["shdi_change"].min()), abs(gdf["shdi_change"].max()))
gdf.plot(column="shdi_change", cmap="RdYlGn", ax=ax, legend=False,
         edgecolor=DARK_NAVY, linewidth=0.3, vmin=-abs_max, vmax=abs_max)
ax.set_title("Change in SHDI (2019 - 2013)", fontsize=14, pad=10)
ax.set_axis_off()

change_top = gdf["shdi_change"].nlargest(3).index.tolist()
change_bot = gdf["shdi_change"].nsmallest(3).index.tolist()
texts = []
for i in change_top + change_bot:
    centroid = gdf.geometry.iloc[i].centroid
    texts.append(ax.text(centroid.x, centroid.y, gdf.loc[i, "region"],
                         fontsize=7, color=WHITE_TEXT, weight="bold"))
adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="-|>",
            color=LIGHT_TEXT, alpha=0.9, lw=1.2,
            mutation_scale=8))

sm = plt.cm.ScalarMappable(cmap="RdYlGn",
                           norm=plt.Normalize(vmin=-abs_max, vmax=abs_max))
cbar = fig.colorbar(sm, ax=ax, orientation="horizontal",
                    fraction=0.03, pad=0.02, aspect=40)
cbar.set_label("SHDI change (2019 - 2013)")

plt.show()

Development losses are geographically concentrated, not randomly scattered. Venezuelan states show the deepest declines (up to -0.067 points), while Colombian and Surinamese regions show the largest improvements (up to +0.045).

## 7. Spatial weights

### 7.1 What is a spatial weights matrix?

To test for spatial clustering formally, we first need to define what "neighbor" means. A **spatial weights matrix** $W$ is an $n \times n$ matrix where each entry $w_{ij}$ encodes the spatial relationship between regions $i$ and $j$. If two regions are neighbors, $w_{ij} > 0$; if not, $w_{ij} = 0$.

The most common approach for polygon data is **contiguity-based weights**:

- **Queen contiguity:** Two regions are neighbors if they share any boundary point (even a single corner). Named after the queen in chess.
- **Rook contiguity:** Two regions are neighbors only if they share an edge. More restrictive than Queen.

### 7.2 Building Queen contiguity weights

PySAL's `Queen.from_dataframe()` builds the weights matrix directly from a GeoDataFrame. After construction, we **row-standardize** the matrix so that each region's neighbor weights sum to 1.

In [ ]:
W = Queen.from_dataframe(gdf)
W.transform = "r"

print(f"Number of regions: {W.n}")
print(f"Min neighbors: {W.min_neighbors}")
print(f"Max neighbors: {W.max_neighbors}")
print(f"Mean neighbors: {W.mean_neighbors:.2f}")
print(f"Islands: {W.islands}")

The Queen contiguity matrix connects 153 regions with an average of 4.93 neighbors each. Two regions have **no neighbors** (islands): San Andres (COL) and Nueva Esparta (VEN) --- both are island territories.

### 7.3 Visualizing the connectivity structure

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

gdf.plot(ax=ax, facecolor="none", edgecolor=GRID_LINE, linewidth=0.5)
plot_spatial_weights(W, gdf, ax=ax)

ax.set_title("Queen contiguity weights", fontsize=14, pad=10)
ax.set_axis_off()

plt.show()

## 8. Global spatial autocorrelation

### 8.1 Moran's I: concept and intuition

**Moran's I** is the most widely used measure of global spatial autocorrelation. Think of it like temperature on a weather map --- if it is hot in one city, nearby cities are likely hot too.

The statistic is defined as:

$$I = \frac{n}{\sum_{i} \sum_{j} w_{ij}} \cdot \frac{\sum_{i} \sum_{j} w_{ij} (x_i - \bar{x})(x_j - \bar{x})}{\sum_{i} (x_i - \bar{x})^2}$$

where $n$ is the number of regions, $w_{ij}$ are the spatial weights, $x_i$ is the value at region $i$, and $\bar{x}$ is the overall mean.

- $I \approx +1$: strong positive spatial autocorrelation (clustering of similar values)
- $I \approx 0$: no spatial pattern (random arrangement)
- $I \approx -1$: strong negative spatial autocorrelation (checkerboard pattern)

### 8.2 Moran's I for HDI

We compute Moran's I with 999 random permutations. A **permutation test** works by randomly shuffling all the SHDI values across the map 999 times --- like dealing cards to random seats.

In [ ]:
moran_2013 = Moran(gdf["shdi2013"], W, permutations=999)
moran_2019 = Moran(gdf["shdi2019"], W, permutations=999)

print(f"SHDI 2013: I = {moran_2013.I:.4f}, p-value = {moran_2013.p_sim:.4f}, "
      f"z-score = {moran_2013.z_sim:.4f}")
print(f"SHDI 2019: I = {moran_2019.I:.4f}, p-value = {moran_2019.p_sim:.4f}, "
      f"z-score = {moran_2019.z_sim:.4f}")
print(f"Expected I (random): {moran_2013.EI:.4f}")

Moran's I is strongly positive and highly significant in both years (2013: I = 0.568, 2019: I = 0.632). Spatial autocorrelation **strengthened** from 2013 to 2019.

### 8.3 Moran scatter plot

The **Moran scatter plot** plots each region's standardized value ($z_i$) against the spatial lag of its neighbors ($Wz_i$). The slope equals Moran's I. The four quadrants: HH (top-right), LL (bottom-left), LH (top-left, spatial outlier), HL (bottom-right, spatial outlier).

In [ ]:
from scipy import stats as scipy_stats

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, moran_obj, year in [
    (axes[0], moran_2013, "2013"),
    (axes[1], moran_2019, "2019"),
]:
    y = gdf[f"shdi{year}"].values
    z = (y - y.mean()) / y.std()
    wz = lag_spatial(W, z)

    ax.scatter(z, wz, color=STEEL_BLUE, s=35, alpha=0.7,
               edgecolors=GRID_LINE, linewidths=0.3, zorder=3)

    slope, intercept, _, _, _ = scipy_stats.linregress(z, wz)
    x_range = np.array([z.min(), z.max()])
    ax.plot(x_range, intercept + slope * x_range, color=WARM_ORANGE,
            linewidth=1.5, zorder=2)

    ax.axhline(0, color=LIGHT_TEXT, linewidth=0.8, alpha=0.5, zorder=1)
    ax.axvline(0, color=LIGHT_TEXT, linewidth=0.8, alpha=0.5, zorder=1)

    xlim, ylim = ax.get_xlim(), ax.get_ylim()
    pad_x = (xlim[1] - xlim[0]) * 0.05
    pad_y = (ylim[1] - ylim[0]) * 0.05
    ax.text(xlim[1] - pad_x, ylim[1] - pad_y, "HH", fontsize=13,
            ha="right", va="top", color=LIGHT_TEXT, alpha=0.5)
    ax.text(xlim[0] + pad_x, ylim[1] - pad_y, "LH", fontsize=13,
            ha="left", va="top", color=LIGHT_TEXT, alpha=0.5)
    ax.text(xlim[0] + pad_x, ylim[0] + pad_y, "LL", fontsize=13,
            ha="left", va="bottom", color=LIGHT_TEXT, alpha=0.5)
    ax.text(xlim[1] - pad_x, ylim[0] + pad_y, "HL", fontsize=13,
            ha="right", va="bottom", color=LIGHT_TEXT, alpha=0.5)

    ax.set_xlabel(f"SHDI {year} (standardized)")
    ax.set_ylabel(f"Spatial lag of SHDI {year}")
    ax.set_title(f"({'a' if year == '2013' else 'b'}) Moran scatter plot "
                 f"\u2014 {year} (I = {moran_obj.I:.4f})")

plt.tight_layout()
plt.show()

## 9. Local spatial autocorrelation (LISA)

### 9.1 From global to local

LISA decomposes the global statistic into a contribution from each individual region.

The local Moran statistic for region $i$ is:

$$I_i = z_i \sum_{j} w_{ij} z_j$$

where $z_i = (x_i - \bar{x}) / s$ is the standardized value at region $i$ and $\sum_{j} w_{ij} z_j$ is its spatial lag. In the code, $x_i$ corresponds to `gdf["shdi2019"]` and $w_{ij}$ to the row-standardized Queen weights `W`.

The four quadrant types: HH (hot spot), LL (cold spot), HL (positive outlier), LH (negative outlier). Significance assessed via permutation tests at $p < 0.10$.

### 9.2 LISA for HDI 2019

In [ ]:
localMoran_2019 = Moran_Local(gdf["shdi2019"], W, permutations=999, seed=12345)
wlag_2019 = lag_spatial(W, gdf["shdi2019"].values)

sig_2019 = localMoran_2019.p_sim < 0.10
q_labels = {1: "HH", 2: "LH", 3: "LL", 4: "HL"}
for q_val, q_name in q_labels.items():
    count = ((localMoran_2019.q == q_val) & sig_2019).sum()
    print(f"  {q_name}: {count}")
print(f"  Not significant: {(~sig_2019).sum()}")

In [ ]:
LISA_COLORS = {1: "#d7191c", 2: "#89cff0", 3: "#2c7bb6", 4: "#fdae61"}

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(14, 6))

ax = axes[0]
slope, intercept, _, _, _ = scipy_stats.linregress(gdf["shdi2019"].values, wlag_2019)

ns_mask = ~sig_2019
ax.scatter(gdf.loc[ns_mask, "shdi2019"], wlag_2019[ns_mask],
           color="#bababa", s=30, alpha=0.4, edgecolors=GRID_LINE,
           linewidths=0.3, label="ns", zorder=2)

for q_val, q_name in q_labels.items():
    mask = (localMoran_2019.q == q_val) & sig_2019
    if mask.any():
        ax.scatter(gdf.loc[mask, "shdi2019"], wlag_2019[mask],
                   color=LISA_COLORS[q_val], s=40, alpha=0.8,
                   edgecolors=GRID_LINE, linewidths=0.3,
                   label=q_name, zorder=3)

x_range = np.array([gdf["shdi2019"].min(), gdf["shdi2019"].max()])
ax.plot(x_range, intercept + slope * x_range, color=WARM_ORANGE,
        linewidth=1.2, zorder=1)

ax.axhline(wlag_2019.mean(), color=GRID_LINE, linewidth=0.8, linestyle="--", zorder=0)
ax.axvline(gdf["shdi2019"].mean(), color=GRID_LINE, linewidth=0.8, linestyle="--", zorder=0)

ax.set_xlabel("SHDI 2019")
ax.set_ylabel("Spatial lag of SHDI 2019")
ax.set_title(f"(a) Moran scatter plot (I = {moran_2019.I:.4f})")

lisa_cluster(localMoran_2019, gdf, p=0.10,
             legend_kwds={"bbox_to_anchor": (0.02, 0.90)}, ax=axes[1])
axes[1].set_facecolor(DARK_NAVY)
axes[1].set_title("(b) LISA clusters (p < 0.10)")

label_idx = []
hh_mask = (localMoran_2019.q == 1) & sig_2019
if hh_mask.any():
    label_idx += gdf.loc[hh_mask, "shdi2019"].nlargest(3).index.tolist()
ll_mask = (localMoran_2019.q == 3) & sig_2019
if ll_mask.any():
    label_idx += gdf.loc[ll_mask, "shdi2019"].nsmallest(3).index.tolist()
hl_mask = (localMoran_2019.q == 4) & sig_2019
if hl_mask.any():
    label_idx.append(gdf.loc[hl_mask, "shdi2019"].idxmax())
lh_mask = (localMoran_2019.q == 2) & sig_2019
if lh_mask.any():
    label_idx.append(gdf.loc[lh_mask, "shdi2019"].idxmin())

texts = [axes[0].text(gdf.loc[i, "shdi2019"], wlag_2019[i], gdf.loc[i, "region"],
         fontsize=7, color=LIGHT_TEXT) for i in label_idx]
adjust_text(texts, ax=axes[0], arrowprops=dict(arrowstyle="-", color=LIGHT_TEXT,
            alpha=0.5, lw=0.5))

texts = [axes[1].text(gdf.geometry.iloc[i].centroid.x, gdf.geometry.iloc[i].centroid.y,
         gdf.loc[i, "region_country"], fontsize=7, color=WHITE_TEXT, weight="bold")
         for i in label_idx]
adjust_text(texts, ax=axes[1], arrowprops=dict(arrowstyle="-|>", color=LIGHT_TEXT,
            alpha=0.9, lw=1.2, mutation_scale=8))

plt.tight_layout()
plt.show()

The 2019 LISA identifies **30 HH regions**, **37 LL regions**, **5 HL outliers**, **1 LH outlier**, and **80 non-significant regions**.

### 9.3 LISA for HDI 2013

In [ ]:
localMoran_2013 = Moran_Local(gdf["shdi2013"], W, permutations=999, seed=12345)
wlag_2013 = lag_spatial(W, gdf["shdi2013"].values)

sig_2013 = localMoran_2013.p_sim < 0.10
for q_val, q_name in q_labels.items():
    count = ((localMoran_2013.q == q_val) & sig_2013).sum()
    print(f"  {q_name}: {count}")
print(f"  Not significant: {(~sig_2013).sum()}")

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(14, 6))

ax = axes[0]
slope, intercept, _, _, _ = scipy_stats.linregress(gdf["shdi2013"].values, wlag_2013)

ns_mask = ~sig_2013
ax.scatter(gdf.loc[ns_mask, "shdi2013"], wlag_2013[ns_mask],
           color="#bababa", s=30, alpha=0.4, edgecolors=GRID_LINE,
           linewidths=0.3, label="ns", zorder=2)

for q_val, q_name in q_labels.items():
    mask = (localMoran_2013.q == q_val) & sig_2013
    if mask.any():
        ax.scatter(gdf.loc[mask, "shdi2013"], wlag_2013[mask],
                   color=LISA_COLORS[q_val], s=40, alpha=0.8,
                   edgecolors=GRID_LINE, linewidths=0.3,
                   label=q_name, zorder=3)

x_range = np.array([gdf["shdi2013"].min(), gdf["shdi2013"].max()])
ax.plot(x_range, intercept + slope * x_range, color=WARM_ORANGE,
        linewidth=1.2, zorder=1)

ax.axhline(wlag_2013.mean(), color=GRID_LINE, linewidth=0.8, linestyle="--", zorder=0)
ax.axvline(gdf["shdi2013"].mean(), color=GRID_LINE, linewidth=0.8, linestyle="--", zorder=0)

ax.set_xlabel("SHDI 2013")
ax.set_ylabel("Spatial lag of SHDI 2013")
ax.set_title(f"(a) Moran scatter plot (I = {moran_2013.I:.4f})")

lisa_cluster(localMoran_2013, gdf, p=0.10,
             legend_kwds={"bbox_to_anchor": (0.02, 0.90)}, ax=axes[1])
axes[1].set_facecolor(DARK_NAVY)
axes[1].set_title("(b) LISA clusters (p < 0.10)")

label_idx = []
hh_mask = (localMoran_2013.q == 1) & sig_2013
if hh_mask.any():
    label_idx += gdf.loc[hh_mask, "shdi2013"].nlargest(3).index.tolist()
ll_mask = (localMoran_2013.q == 3) & sig_2013
if ll_mask.any():
    label_idx += gdf.loc[ll_mask, "shdi2013"].nsmallest(3).index.tolist()
hl_mask = (localMoran_2013.q == 4) & sig_2013
if hl_mask.any():
    label_idx.append(gdf.loc[hl_mask, "shdi2013"].idxmax())
lh_mask = (localMoran_2013.q == 2) & sig_2013
if lh_mask.any():
    label_idx.append(gdf.loc[lh_mask, "shdi2013"].idxmin())

texts = [axes[0].text(gdf.loc[i, "shdi2013"], wlag_2013[i], gdf.loc[i, "region"],
         fontsize=7, color=LIGHT_TEXT) for i in label_idx]
adjust_text(texts, ax=axes[0], arrowprops=dict(arrowstyle="-", color=LIGHT_TEXT,
            alpha=0.5, lw=0.5))

texts = [axes[1].text(gdf.geometry.iloc[i].centroid.x, gdf.geometry.iloc[i].centroid.y,
         gdf.loc[i, "region_country"], fontsize=7, color=WHITE_TEXT, weight="bold")
         for i in label_idx]
adjust_text(texts, ax=axes[1], arrowprops=dict(arrowstyle="-|>", color=LIGHT_TEXT,
            alpha=0.9, lw=1.2, mutation_scale=8))

plt.tight_layout()
plt.show()

The 2013 LISA identifies **31 HH**, **29 LL**, **5 HL**, **0 LH**, and **88 non-significant**. The LL cluster expanded from 29 to 37 regions by 2019.

### 9.4 Comparing LISA clusters across time

In [ ]:
sig_2013 = localMoran_2013.p_sim < 0.10
sig_2019 = localMoran_2019.p_sim < 0.10
q_labels = {1: "HH", 2: "LH", 3: "LL", 4: "HL"}

labels_2013 = ["ns" if not sig_2013[i] else q_labels[localMoran_2013.q[i]]
               for i in range(len(gdf))]
labels_2019 = ["ns" if not sig_2019[i] else q_labels[localMoran_2019.q[i]]
               for i in range(len(gdf))]

transition_df = pd.crosstab(
    pd.Series(labels_2013, name="2013"),
    pd.Series(labels_2019, name="2019")
)
print(transition_df.to_string())

Strong cluster persistence. Of the 31 HH regions in 2013, **27 remained HH** (87%). **17 regions** joined the LL cluster from non-significant by 2019.

## 10. Space-time dynamics

### 10.1 Directional Moran scatter plot

A **directional Moran scatter plot** shows the movement vector for each region from its 2013 position to its 2019 position. We standardize both years using the pooled mean and standard deviation.

In [ ]:
mean_all = np.mean(np.concatenate([gdf["shdi2013"].values, gdf["shdi2019"].values]))
std_all = np.std(np.concatenate([gdf["shdi2013"].values, gdf["shdi2019"].values]))
z_2013 = (gdf["shdi2013"].values - mean_all) / std_all
z_2019 = (gdf["shdi2019"].values - mean_all) / std_all

wz_2013 = lag_spatial(W, z_2013)
wz_2019 = lag_spatial(W, z_2019)

fig, ax = plt.subplots(figsize=(9, 8))

for i in range(len(gdf)):
    ax.annotate("", xy=(z_2019[i], wz_2019[i]),
                xytext=(z_2013[i], wz_2013[i]),
                arrowprops=dict(arrowstyle="->", color=STEEL_BLUE,
                                alpha=0.5, lw=0.8))

ax.scatter(z_2013, wz_2013, color=WARM_ORANGE, s=20, alpha=0.6,
           label="2013", zorder=4)
ax.scatter(z_2019, wz_2019, color=TEAL, s=20, alpha=0.6,
           label="2019", zorder=4)

ax.axhline(0, color=GRID_LINE, linewidth=1)
ax.axvline(0, color=GRID_LINE, linewidth=1)
ax.set_xlabel("SHDI (standardized)")
ax.set_ylabel("Spatial lag of SHDI")
ax.set_title("Directional Moran scatter plot: movements from 2013 to 2019")
ax.legend()

plt.show()

In [ ]:
q_2013 = np.where((z_2013 >= 0) & (wz_2013 >= 0), "HH",
          np.where((z_2013 < 0) & (wz_2013 >= 0), "LH",
          np.where((z_2013 < 0) & (wz_2013 < 0), "LL", "HL")))

q_2019 = np.where((z_2019 >= 0) & (wz_2019 >= 0), "HH",
          np.where((z_2019 < 0) & (wz_2019 >= 0), "LH",
          np.where((z_2019 < 0) & (wz_2019 < 0), "LL", "HL")))

transition_moran = pd.crosstab(
    pd.Series(q_2013, name="2013"),
    pd.Series(q_2019, name="2019")
)
print(transition_moran.to_string())

stayed = (q_2013 == q_2019).sum()
moved = (q_2013 != q_2019).sum()
print(f"\nStayed in same quadrant: {stayed} ({stayed/len(gdf)*100:.1f}%)")
print(f"Moved to different quadrant: {moved} ({moved/len(gdf)*100:.1f}%)")

**95 regions (62.1%)** remained in the same quadrant, while **58 (37.9%)** crossed boundaries. HH is most stable (76%), followed by LL (62%).

### 10.2 Country focus: Venezuela vs Bolivia

Venezuela and Bolivia offer a stark contrast. We isolate them in the directional Moran scatter plot to compare their movement vectors.

In [ ]:
ven_mask = gdf["country"] == "Venezuela"
bol_mask = gdf["country"] == "Bolivia"

all_z = np.concatenate([z_2013, z_2019])
all_wz = np.concatenate([wz_2013, wz_2019])
pad = 0.3
shared_xlim = (all_z.min() - pad, all_z.max() + pad)
shared_ylim = (all_wz.min() - pad, all_wz.max() + pad)

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(16, 7))

for ax, mask, title in [
    (axes[0], bol_mask, "(a) Bolivia"),
    (axes[1], ven_mask, "(b) Venezuela"),
]:
    for i in range(len(gdf)):
        ax.annotate("", xy=(z_2019[i], wz_2019[i]),
                    xytext=(z_2013[i], wz_2013[i]),
                    arrowprops=dict(arrowstyle="->", color=GRID_LINE,
                                    alpha=0.15, lw=0.5))
    ax.scatter(z_2013, wz_2013, color=GRID_LINE, s=10, alpha=0.15, zorder=2)
    ax.scatter(z_2019, wz_2019, color=GRID_LINE, s=10, alpha=0.15, zorder=2)

    for i in gdf.index[mask]:
        ax.annotate("", xy=(z_2019[i], wz_2019[i]),
                    xytext=(z_2013[i], wz_2013[i]),
                    arrowprops=dict(arrowstyle="->", color=STEEL_BLUE,
                                    alpha=0.7, lw=1.0))
    ax.scatter(z_2013[mask], wz_2013[mask], color=WARM_ORANGE, s=30,
               alpha=0.8, edgecolors=GRID_LINE, linewidths=0.3,
               label="2013", zorder=5)
    ax.scatter(z_2019[mask], wz_2019[mask], color=TEAL, s=30,
               alpha=0.8, edgecolors=GRID_LINE, linewidths=0.3,
               label="2019", zorder=5)

    texts = []
    for i in gdf.index[mask]:
        texts.append(ax.text(z_2019[i], wz_2019[i], gdf.loc[i, "region"],
                             fontsize=7, color=LIGHT_TEXT))
    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="-", color=LIGHT_TEXT,
                alpha=0.5, lw=0.5))

    ax.axhline(0, color=GRID_LINE, linewidth=1, zorder=1)
    ax.axvline(0, color=GRID_LINE, linewidth=1, zorder=1)
    ax.set_xlim(shared_xlim)
    ax.set_ylim(shared_ylim)

    ox = (shared_xlim[1] - shared_xlim[0]) * 0.05
    oy = (shared_ylim[1] - shared_ylim[0]) * 0.05
    for lbl, ha, va, x, y in [
        ("HH", "right", "top", shared_xlim[1] - ox, shared_ylim[1] - oy),
        ("LH", "left", "top", shared_xlim[0] + ox, shared_ylim[1] - oy),
        ("LL", "left", "bottom", shared_xlim[0] + ox, shared_ylim[0] + oy),
        ("HL", "right", "bottom", shared_xlim[1] - ox, shared_ylim[0] + oy),
    ]:
        ax.text(x, y, lbl, fontsize=14, ha=ha, va=va,
                color=LIGHT_TEXT, alpha=0.6)

    ax.set_xlabel("SHDI (standardized)")
    ax.set_ylabel("Spatial lag of SHDI")
    ax.set_title(title)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
for country, mask in [("Venezuela", ven_mask), ("Bolivia", bol_mask)]:
    n = mask.sum()
    mean_change = gdf.loc[mask, "shdi_change"].mean()
    min_change = gdf.loc[mask, "shdi_change"].min()
    max_change = gdf.loc[mask, "shdi_change"].max()
    q13 = q_2013[mask]
    q19 = q_2019[mask]
    stayed = (q13 == q19).sum()
    moved = (q13 != q19).sum()
    print(f"\n{country} ({n} regions):")
    print(f"  Mean SHDI change: {mean_change:+.4f}")
    print(f"  Range: [{min_change:+.4f}, {max_change:+.4f}]")
    print(f"  Quadrant stability: {stayed} stayed, {moved} moved")
    print(f"  2013 quadrants: {', '.join(f'{q}={c}' for q, c in zip(*np.unique(q13, return_counts=True)))}")
    print(f"  2019 quadrants: {', '.join(f'{q}={c}' for q, c in zip(*np.unique(q19, return_counts=True)))}")

Venezuela's 24 regions experienced the most dramatic downward shift in the dataset, with a mean SHDI change of -0.065. In 2013, 13 regions were in HH. By 2019, none remained --- 21 of 24 (88%) crossed quadrant boundaries, ending in LL.

Bolivia tells the opposite story. Its 9 regions started in the lower-left (8 in LL, 1 in LH) and moved modestly rightward. Mean SHDI change was +0.033. Seven of 9 regions (78%) remained in the same quadrant.

## 11. Discussion

**Spatial autocorrelation in South American human development is strong and persistent.** Global Moran's I increased from 0.568 in 2013 to 0.632 in 2019 (both p = 0.001), indicating that the spatial clustering of development levels strengthened over the period. This means the development gap between prosperous and lagging regions is not only large but spatially structured --- high-development regions form a contiguous band across the Southern Cone, while low-development regions form an equally contiguous band across the Amazon basin and northern South America.

The LISA analysis pinpoints these clusters with precision. In 2019, 30 regions form a significant HH cluster (high development surrounded by high-development neighbors) and 37 regions form a significant LL cluster (low development surrounded by low-development neighbors). The LL cluster expanded from 29 to 37 regions between 2013 and 2019, driven primarily by Venezuela's economic crisis and its spillover effects on neighboring regions. The HH cluster remained stable (31 to 30), with 87% persistence --- a sign that prosperity corridors in the Southern Cone are structurally entrenched.

The space-time analysis reveals that 62% of regions stayed in the same Moran scatter plot quadrant, but the 38% that moved tell an important story. The most concerning transitions are the 10 regions that moved from HH to LL and the 17 previously non-significant regions that joined the LL LISA cluster. These movements are concentrated in Venezuela and its neighbors, illustrating how economic shocks can propagate spatially.

The **Venezuela--Bolivia comparison** crystallizes the two forces shaping South America's spatial development landscape. Venezuela's 24 regions collapsed nearly uniformly (mean SHDI change of -0.065, with 88% crossing quadrant boundaries), transforming a country that was largely in the HH quadrant in 2013 into one almost entirely in the LL quadrant by 2019. Bolivia's 9 regions, starting from a much lower base, improved steadily (+0.033) with 78% quadrant stability. These divergent trajectories illustrate that spatial clusters are not static: they can expand rapidly through crisis-driven contagion (Venezuela pulling its neighbors downward) or contract slowly through sustained internal improvement (Bolivia gradually lifting its regions rightward in the Moran scatter plot). The fact that Venezuela's decline was spatially contagious --- dragging down the spatial lags of neighboring Colombian and Brazilian border regions --- while Bolivia's improvement remained spatially contained underscores an asymmetry: negative shocks propagate faster and farther across borders than positive ones.

For policy, these findings suggest that **spatially targeted interventions** may be more effective than uniform national programs. The persistent LL clusters represent development traps where a region's own conditions are reinforced by the equally poor conditions of its neighbors. Breaking these traps may require coordinated cross-regional or cross-border programs that address the spatial dimension of underdevelopment. Bolivia's experience suggests that broad-based national improvement can lift all regions, but escaping the low-development spatial cluster may require the additional step of improving neighbors' conditions simultaneously --- a challenge that calls for cross-border cooperation.

## 12. Summary and next steps

**Key takeaways:**

- **Method insight:** ESDA reveals spatial patterns invisible in aspatial analysis. The same dataset that shows a modest aggregate improvement (+0.005 SHDI) conceals a deepening spatial divide --- Moran's I increased from 0.568 to 0.632, meaning spatial clustering strengthened between 2013 and 2019.
- **Data insight:** 30 HH and 37 LL regions form statistically significant clusters at the 10% level. The LL cluster expanded by 8 regions (from 29 to 37), while the HH cluster remained stable. Cluster persistence is high: 87% for HH and 62% for LL, indicating entrenched spatial inequality.
- **Country insight:** Venezuela and Bolivia illustrate contrasting development dynamics. Venezuela's 24 regions collapsed nearly uniformly (mean -0.065), with 88% crossing quadrant boundaries from the upper to the lower portion of the Moran scatter plot. Bolivia's 9 regions improved steadily (+0.033) with 78% quadrant stability, showing broad-based gains that have not yet been large enough to escape the LL spatial cluster.
- **Limitation:** Queen contiguity assumes shared borders, which excludes island territories (San Andres, Nueva Esparta) and may not capture cross-water economic linkages. With only two time periods (2013 and 2019), we cannot distinguish permanent structural clusters from temporary effects of the Venezuelan crisis. The p = 0.10 significance threshold is relatively permissive.
- **Next step:** Extend the analysis with spatial regression models (spatial lag and spatial error models) to test whether a region's development is directly influenced by its neighbors' development, or whether the clustering is driven by shared underlying factors. Bivariate LISA could reveal whether income clusters coincide with education clusters. Adding more time periods (2000--2019) from the full Global Data Lab series would enable Spatial Markov chain analysis of cluster transition probabilities.

## 13. Exercises

1. **Income clusters.** Repeat the LISA analysis for the income index (`incindex2019`) instead of SHDI. Are income clusters in the same locations as HDI clusters? How many regions belong to both an income LL and an HDI LL cluster?

2. **Alternative weights.** Build k-nearest neighbors weights (`KNN` from `libpysal.weights`) with $k = 5$ and Rook contiguity (`Rook` from `libpysal.weights`) instead of Queen contiguity. How does Moran's I change under each specification? Does the KNN approach resolve the island problem?

3. **Bivariate Moran.** Use [`Moran_BV`](https://pysal.org/esda/generated/esda.Moran_BV.html) from esda to compute the bivariate Moran's I between education and income indices. Are regions with high education surrounded by regions with high income, or are the two dimensions spatially independent?

4. **Spatial autocorrelation of change.** Compute Moran's I for `shdi_change` instead of the level variables. Is the *change* in SHDI between 2013 and 2019 itself spatially clustered? Compare the result with the change choropleth from Section 6.2. Hint: `Moran(gdf["shdi_change"], W, permutations=999)`.

5. **Component-level Moran's I.** Compute Moran's I for the health, education, and income indices separately in both 2013 and 2019. Which component shows the strongest spatial autocorrelation? Does the income index --- which declined in 46% of regions --- show a different spatial pattern than health or education?

6. **Multiple testing sensitivity.** Re-run the 2019 LISA analysis at $p < 0.05$ instead of $p < 0.10$. How many HH and LL regions survive the stricter threshold? Research the Bonferroni correction ($0.05 / 153 \approx 0.0003$) and the False Discovery Rate (FDR) procedure --- how would these affect the cluster counts?

7. **Neighbor count distribution.** Plot a histogram of the number of neighbors per region from the Queen weights matrix (use `W.cardinalities`). What is the shape of the distribution? Which regions have the most and fewest neighbors, and why?

8. **Is the Moran's I increase significant?** Moran's I rose from 0.568 to 0.632 between 2013 and 2019. But does this difference pass a significance test? Try a bootstrap approach: pool the 2013 and 2019 SHDI values, randomly assign them to the two periods 999 times, and compute the difference in Moran's I each time. Where does the observed difference (0.064) fall in the bootstrap distribution?

9. **Moran's I excluding Venezuela.** Recompute Moran's I for 2013 and 2019 after dropping Venezuela's 24 regions (rebuild the Queen weights on the subset GeoDataFrame). Does the increase in spatial autocorrelation survive? If not, the "deepening spatial divide" may be driven by a single country's crisis rather than a continent-wide trend.

10. **LISA significance map.** Create a choropleth map coloring each region by its LISA p-value (`localMoran_2019.p_sim`) using a sequential colormap. How many regions have $p < 0.01$ vs $p < 0.05$ vs $p < 0.10$? Are the deeply significant regions ($p < 0.01$) concentrated in the same locations as the cluster map from Section 9.2?

## 14. References

1. [Anselin, L. (1995). Local Indicators of Spatial Association --- LISA. *Geographical Analysis*, 27(2), 93--115.](https://doi.org/10.1111/j.1538-4632.1995.tb00338.x)
2. [Smits, J. and Permanyer, I. (2019). The Subnational Human Development Database. *Scientific Data*, 6, 190038.](https://doi.org/10.1038/sdata.2019.38)
3. [Rey, S. J. and Anselin, L. (2007). PySAL: A Python Library of Spatial Analytical Methods. *Review of Regional Studies*, 37(1), 5--27.](https://rrs.scholasticahq.com/article/8285)
4. [Global Data Lab --- Subnational Human Development Index](https://globaldatalab.org/shdi/)
5. [PySAL ESDA documentation](https://pysal.org/esda/)
6. [splot documentation](https://splot.readthedocs.io/)
7. [Mendez, C. (2026). Pooled PCA for Building Development Indicators Across Time.](https://carlos-mendez.org/post/python_pca2/)
8. [Mendez, C. and Gonzales, E. (2021). Human Capital Constraints, Spatial Dependence, and Regionalization in Bolivia. *Economia*, 44(87).](https://carlos-mendez.org/publication/20210318-economia/)
9. [Mendez, C. (2026). Monitoring Regional Development with Python.](https://carlos-mendez.org/post/python_monitor_regional_development/)